In [ ]:
import os
from pathlib import Path
PROJECT_ROOT = Path((os.environ.get("TEAM18_PROJECT_ROOT") or next(
    str(p) for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "run.py").is_file() and (p / "src" / "steam_reviews").is_dir()
))).resolve()
DATA_DIR = PROJECT_ROOT / "data"
EDA_OUTPUT = PROJECT_ROOT / "runs" / "data_preparation"
EDA_OUTPUT.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import gc

In [ ]:
pd.set_option("display.max_rows", None)

In [ ]:
from pathlib import Path
import duckdb

BASE0 = str(PROJECT_ROOT / 'data/interim/reviews_analysis_base/**/*.parquet')
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE0}', hive_partitioning=true)")

# import base dataset

In [ ]:
BASE = Path(
    str(PROJECT_ROOT / 'data/interim/reviews_analysis_base')
)

files = list(BASE.rglob("*.parquet"))

print(f"Found {len(files)} parquet files")

df_base = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(df_base.shape)
gc.collect()

In [ ]:
df_base.head()


In [ ]:
df_base.columns.tolist()

## kaggle raw data

In [ ]:
df_kaggle = pd.read_csv(str(PROJECT_ROOT / 'data/raw/reviews/reviews-1-115.csv'))
df_kaggle.shape

In [ ]:
df_kaggle.head()

In [ ]:
df_kaggle.columns

In [ ]:
set(df_kaggle.columns) ^ set(df_base.columns)

## filter columns

In [ ]:
keep_cols = [
    'steamid', 'appid', 'voted_up', 'votes_up', 'votes_funny',
    'weighted_vote_score', 'playtime_forever', 'playtime_at_review',
    'num_games_owned', 'num_reviews', 'review',
    'unix_timestamp_created', 'unix_timestamp_updated',
    'game_name', 'genres', 'release_date'
]

df_base = df_base[keep_cols]
df_base.shape

In [ ]:
df_base.head()

# EDA

## process genres

In [ ]:
genres = (
    df_base["genres"]
    .dropna()
    .explode()
    .dropna()
)

print("Number of unique genres:", genres.nunique())

In [ ]:
sorted(genres.unique())

In [ ]:
genre_counts = genres.value_counts()

genre_counts

### combine genres with similar meanings

In [ ]:
df = df_base.copy()

print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

In [ ]:
genre_map = {
    "Free To Play": "Free to Play",
    "RPG": "Role-playing (RPG)",
    "Role-playing": "Role-playing (RPG)",
    "Sport": "Sports",
    "Platform": "Platformer",
    "Simulator": "Simulation",
    'Card Game': 'Card & Board Game',
    'Real Time Strategy (RTS)': 'Real-time',
    'Turn-based strategy (TBS)': 'Turn-based'
}

df["genres_clean"] = df["genres"].apply(
    lambda xs: [genre_map.get(x, x) for x in xs]
    if xs is not None else xs
)

genres_clean = (
    df["genres_clean"]
    .dropna()
    .explode()
    .dropna()
)

def clean_genres(xs):
    if xs is None:
        return xs

    cleaned = [genre_map.get(x, x) for x in xs]
    return list(dict.fromkeys(cleaned))

df["genres_clean"] = df["genres_clean"].apply(clean_genres)

print(genres_clean.nunique())
print(sorted(genres_clean.unique()))

In [ ]:
df['genres_clean']

In [ ]:
game_genres = (
    df[["appid", "genres_clean"]]
    .drop_duplicates("appid")
    .reset_index(drop=True)
)

game_genres.head()

In [ ]:
genre_game_counts = (
    game_genres
    .explode("genres_clean")["genres_clean"]
    .value_counts()
)

genre_game_counts

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Define common and rare genres
MIN_GAMES = 50

keep_genres = set(
    genre_game_counts[genre_game_counts >= MIN_GAMES].index
)

rare_genres = set(
    genre_game_counts[genre_game_counts < MIN_GAMES].index
)

print("Genres kept:", len(keep_genres))
print("Rare genres collapsed into Other:", len(rare_genres))

print("\nKept genres:")
print(sorted(keep_genres))

print("\nRare genres:")
print(sorted(rare_genres))

In [ ]:
def collapse_genres(xs):
    # Handle missing genres
    if not isinstance(xs, (list, tuple, set, np.ndarray)):
        return []

    # Keep common genres
    result = [g for g in xs if g in keep_genres]

    # If there is at least one rare genre, add Other
    if any(g in rare_genres for g in xs):
        result.append("Other")

    return list(dict.fromkeys(result))

df["genres_model"] = df["genres_clean"].apply(collapse_genres)

In [ ]:
# 3. Multi-hot encoding
model_genres = sorted(keep_genres) + ["Other"]

mlb = MultiLabelBinarizer(
    classes=model_genres,
    sparse_output=True
)

genre_sparse = mlb.fit_transform(df["genres_model"])

In [ ]:
# 4. Convert to sparse DataFrame
genre_dummies = pd.DataFrame.sparse.from_spmatrix(
    genre_sparse,
    index=df.index,
    columns=[f"genre_{g}" for g in model_genres]
)
# 5. Add genre dummy variables to df
df = pd.concat(
    [df, genre_dummies],
    axis=1
)

In [ ]:

df.drop(
    columns=["genres", "genres_clean", "genres_model"],
    inplace=True
)
print(df.shape)

In [ ]:
df.columns.tolist()

## null summary

In [ ]:
# Null summary - works with mixed dense + sparse columns

missing_n = pd.Series({
    col: int(df[col].isna().sum())
    for col in df.columns
})

null_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": missing_n,
    "missing_pct": missing_n / len(df) * 100
})

null_summary = null_summary.sort_values(
    "missing_pct",
    ascending=False
)

display(null_summary)

In [ ]:
same_missing = (
    df["game_name"].isna() ==
    df["release_date"].isna()
)

print("Same missing pattern:",
      f"{same_missing.mean() * 100:.4f}%")

print(
    "Rows where only one is missing:",
    (~same_missing).sum()
)

# new data

In [ ]:
df_new_raw = pd.read_parquet(str(PROJECT_ROOT / 'data/interim/negative_reviews_new.parquet'))
df_new_raw.shape

In [ ]:
df_new_raw.head()

In [ ]:
df_new_raw.columns.tolist()

In [ ]:
# Null summary - works with mixed dense + sparse columns

missing_n = pd.Series({
    col: int(df_new_raw[col].isna().sum())
    for col in df_new_raw.columns
})

null_summary = pd.DataFrame({
    "dtype": df_new_raw.dtypes.astype(str),
    "missing_n": missing_n,
    "missing_pct": missing_n / len(df_new_raw) * 100
})

null_summary = null_summary.sort_values(
    "missing_pct",
    ascending=False
)

display(null_summary)

In [ ]:
TARGET_COLUMNS = [
    "still_playing_d7",
    "still_playing_d30",
    "still_playing_d90",
    "still_playing_d180",
]

BEHAVIOR_COLUMNS = [
    "hours_forever_at_snapshot",
    "hours_last_2w_at_snapshot",
    "last_played_utc",
    "no_play_after_review",
]

# Record missing-value counts before processing for verification.
null_counts_before = (
    df_new_raw[
        TARGET_COLUMNS
        + ["game", "review_length_chars"]
        + BEHAVIOR_COLUMNS
    ]
    .isna()
    .sum()
)

display(null_counts_before)

In [ ]:
# Create the missing-value indicator before imputation.
review_length_missing = df_new_raw["review_length_chars"].isna()

# Calculate median review length by language.
language_length_medians = (
    df_new_raw.loc[
        ~review_length_missing,
        ["language", "review_length_chars"]
    ]
    .groupby("language", observed=True)["review_length_chars"]
    .median()
)

global_length_median = df_new_raw["review_length_chars"].median()

# Generate fill values only for the 142 missing rows, avoiding an intermediate Series for more than 16 million rows.
review_length_fill_values = (
    df_new_raw.loc[review_length_missing, "language"]
    .map(language_length_medians)
    .fillna(global_length_median)
)

df_new_raw.loc[
    review_length_missing,
    "review_length_chars"
] = review_length_fill_values

print(
    "Remaining review_length_chars nulls:",
    df_new_raw["review_length_chars"].isna().sum()
)

In [ ]:
df_new_raw["game"] = df_new_raw["game"].fillna("Unknown")
print(df_new_raw.shape)

In [ ]:
train_start = pd.Timestamp("2010-10-15", tz="UTC")
val_start   = pd.Timestamp("2022-01-01", tz="UTC")
test_start  = pd.Timestamp("2023-01-01", tz="UTC")
# training set size
print(f"training set size: {len(df_new_raw.query('@train_start <= review_created_utc < @val_start'))}")
# validation set size
print(f"validation set size: {len(df_new_raw.query('@val_start <= review_created_utc < @test_start'))}")
# testing set size
print(f"testing set size: {len(df_new_raw.query('review_created_utc >= @test_start'))}")

In [ ]:
df_new_raw.to_parquet(EDA_OUTPUT / 'negative_reviews_cleaned.parquet')

In [ ]:
df_cleaned = pd.read_parquet(EDA_OUTPUT / 'negative_reviews_cleaned.parquet')
print(df_cleaned.shape)

In [ ]:
df_cleaned.head()

In [ ]:
df_nlp_raw = pd.read_parquet(str(PROJECT_ROOT / 'data/processed/nlp/negative_reviews_2023_text_focus5.parquet'))
df_nlp_raw.shape

In [ ]:
df_nlp_raw.head()

In [ ]:
df_nlp_scored = pd.read_parquet(str(PROJECT_ROOT / 'data/processed/nlp/nlp_factor_scores_2023.parquet'))
df_nlp_scored.shape

In [ ]:
df_nlp_scored.head()

## merge data

In [ ]:
# Composite join key
join_keys = ["appid", "recommendationid"]

# 1. Validate key uniqueness
assert not df_cleaned.duplicated(join_keys).any(), \
    "df_cleaned contains duplicate (appid, recommendationid) keys"

assert not df_nlp_scored.duplicated(join_keys).any(), \
    "df_nlp_scored contains duplicate (appid, recommendationid) keys"

# 2. Select only NLP columns not already present in df_cleaned
new_nlp_cols = [
    col for col in df_nlp_scored.columns
    if col not in df_cleaned.columns
]

nlp_for_merge = df_nlp_scored[
    join_keys + new_nlp_cols
]

# 3. Left join: preserve every row in df_cleaned
df_merged = df_cleaned.merge(
    nlp_for_merge,
    on=join_keys,
    how="left",
    validate="one_to_one",
    indicator=True,
)

# 4. Merge audit
print("Original rows:", len(df_cleaned))
print("Merged rows:", len(df_merged))
print("\nMerge status:")
print(df_merged["_merge"].value_counts(dropna=False))

# 5. Remove audit column
df_merged = df_merged.drop(columns="_merge")

# 6. Final validation
assert len(df_merged) == len(df_cleaned), \
    "Merge changed the number of review rows"

assert not df_merged.columns.duplicated().any(), \
    "Merged table contains duplicate column names"

print("\nFinal shape:", df_merged.shape)
print("Duplicate columns:", df_merged.columns.duplicated().sum())

In [ ]:
df_merged.head()

In [ ]:
df_merged.columns.tolist()

In [ ]:
df_merged.to_parquet(EDA_OUTPUT / 'negative_reviews_merged.parquet')

In [ ]:
df_merged = pd.read_parquet(EDA_OUTPUT / 'negative_reviews_merged.parquet')
print(df_merged.shape)

In [ ]:
df_merged.describe()

# Kmeans

## find out customer segementation for negative reviews

### Persona model

In [ ]:
df_segement = df_merged.copy()
gc.collect()

In [ ]:
df_segement["log_hours_at_review"] = np.log1p(
    df_segement["hours_at_review"]
)

df_segement["log_num_games_owned"] = np.log1p(
    df_segement["num_games_owned"]
)

df_segement["log_author_num_reviews"] = np.log1p(
    df_segement["author_num_reviews"]
)

df_segement["log_review_length"] = np.log1p(
    df_segement["review_length_chars"]
)

In [ ]:
import random
from itertools import combinations

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 20261005
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# K search
K_CANDIDATES = range(2, 9)

# Large-data controls
K_SEARCH_SAMPLE_SIZE = 100_000
METRIC_SAMPLE_SIZE = 20_000
SILHOUETTE_SAMPLE_SIZE = 5_000
VISUALIZATION_SAMPLE_SIZE = 15_000

# Final MiniBatchKMeans
CHUNK_SIZE = 50_000
FULL_TRAINING_EPOCHS = 2

# Set an integer such as 4 to override automatic selection.
SELECTED_K = None

# Primary analysis uses unedited reviews because NLP may reflect updated text.
FILTER_EDITED_REVIEWS = True

rng = np.random.default_rng(RANDOM_SEED)

print("Rows in df_segement:", f"{len(df_segement):,}")

In [ ]:
domain_groups = {
    "domain_technical": [
        "p_tech_performance",
        "p_tech_stability",
        "p_tech_functionality",
        "p_tech_save_integrity",
        "p_tech_network_responsiveness",
        "p_tech_network_session_access",
        "p_tech_platform_compatibility",
    ],
    "domain_gameplay": [
        "p_gameplay_core_loop",
        "p_gameplay_balance",
        "p_gameplay_challenge",
        "p_gameplay_randomness",
    ],
    "domain_progression": [
        "p_progression_effort_reward",
    ],
    "domain_content": [
        "p_content_scope_variety",
        "p_content_narrative",
        "p_content_world_level",
    ],
    "domain_social": [
        "p_social_cheating",
        "p_social_skill_matchmaking",
        "p_social_match_availability",
        "p_social_conduct",
        "p_social_sanctions",
    ],
    "domain_ux": [
        "p_ux_interface",
        "p_ux_controls",
        "p_ux_onboarding",
        "p_ux_localization",
        "p_ux_accessibility",
    ],
    "domain_business": [
        "p_business_price_value",
        "p_business_paid_advantage",
        "p_business_paywall",
        "p_business_sales_pressure",
        "p_business_account_security",
    ],
    "domain_service": [
        "p_service_maintenance",
        "p_service_marketing_trust",
        "p_service_support",
        "p_service_release_readiness",
        "p_service_update_regression",
    ],
}

required_probability_columns = sorted({
    column
    for columns in domain_groups.values()
    for column in columns
})

missing_probability_columns = sorted(
    set(required_probability_columns) - set(df_segement.columns)
)

if missing_probability_columns:
    raise KeyError(
        f"Missing NLP probability columns: {missing_probability_columns}"
    )

# Validate probability ranges before aggregation.
probability_audit = []

for column in required_probability_columns:
    values = pd.to_numeric(df_segement[column], errors="coerce")

    out_of_range = (
        values.notna()
        & ((values < 0) | (values > 1))
    ).sum()

    probability_audit.append({
        "feature": column,
        "missing_rate": values.isna().mean(),
        "out_of_range": int(out_of_range),
    })

probability_audit = pd.DataFrame(probability_audit)

display(
    probability_audit.sort_values(
        ["out_of_range", "missing_rate"],
        ascending=[False, False],
    ).head(10)
)

if probability_audit["out_of_range"].sum() > 0:
    raise ValueError(
        "Some NLP probabilities fall outside [0, 1]. "
        "Investigate before clustering."
    )

# Maximum probability within each complaint domain.
for new_column, source_columns in domain_groups.items():
    df_segement[new_column] = (
        df_segement[source_columns]
        .max(axis=1, skipna=True)
        .astype("float32")
    )

In [ ]:
log_transform_map = {
    "hours_at_review": "log_hours_at_review",
    "num_games_owned": "log_num_games_owned",
    "author_num_reviews": "log_author_num_reviews",
    "review_length_chars": "log_review_length",
    "n_factors_flagged": "log_n_factors_flagged",
}

missing_base_columns = sorted(
    set(log_transform_map) - set(df_segement.columns)
)

if missing_base_columns:
    raise KeyError(
        f"Missing user-profile columns: {missing_base_columns}"
    )

negative_value_audit = []

for source_column, new_column in log_transform_map.items():
    values = pd.to_numeric(
        df_segement[source_column],
        errors="coerce",
    )

    negative_count = int((values < 0).sum())

    negative_value_audit.append({
        "feature": source_column,
        "negative_values": negative_count,
        "missing_rate": values.isna().mean(),
    })

    # Negative values are treated as invalid/missing, not silently clipped to 0.
    valid_values = values.mask(values < 0)

    df_segement[new_column] = (
        np.log1p(valid_values)
        .astype("float32")
    )

negative_value_audit = pd.DataFrame(negative_value_audit)
display(negative_value_audit)

In [ ]:
user_features = [
    "log_hours_at_review",
    "log_num_games_owned",
    "log_author_num_reviews",
    "log_review_length",
    # "still_playing_d7",
    "still_playing_d30",
    # "still_playing_d90",
    # "still_playing_d180",
    "votes_up",
    "weighted_vote_score",
    # "comment_count"
]

complaint_features = [
    "domain_technical",
    "domain_gameplay",
    "domain_progression",
    "domain_content",
    "domain_social",
    "domain_ux",
    "domain_business",
    "domain_service",
    "log_n_factors_flagged",
]

cluster_features = user_features + complaint_features 

# Robustly interpret the edited field.
if FILTER_EDITED_REVIEWS and "edited" in df_segement.columns:
    if pd.api.types.is_bool_dtype(df_segement["edited"]):
        edited_flag = df_segement["edited"].fillna(False)
    else:
        edited_flag = (
            df_segement["edited"]
            .astype("string")
            .str.lower()
            .isin(["true", "1", "yes"])
        )
else:
    edited_flag = pd.Series(
        False,
        index=df_segement.index,
    )

# Require at least one nonmissing cluster feature.
has_any_feature = (
    df_segement[cluster_features]
    .notna()
    .any(axis=1)
)

eligible_mask = (~edited_flag) & has_any_feature
eligible_positions = np.flatnonzero(eligible_mask.to_numpy())

print("Total rows:", f"{len(df_segement):,}")
print("Eligible rows:", f"{len(eligible_positions):,}")
print(
    "Eligible share:",
    f"{len(eligible_positions) / len(df_segement):.2%}",
)
print("Edited rows excluded:", f"{edited_flag.sum():,}")

if len(eligible_positions) < 1_000:
    raise ValueError(
        "Too few eligible observations for meaningful clustering."
    )

In [ ]:
search_sample_size = min(
    K_SEARCH_SAMPLE_SIZE,
    len(eligible_positions),
)

search_positions = rng.choice(
    eligible_positions,
    size=search_sample_size,
    replace=False,
)

search_frame = (
    df_segement.iloc[search_positions]
    [cluster_features]
    .copy()
)

print("K-search sample:", f"{len(search_frame):,}")

In [ ]:
user_imputer = SimpleImputer(strategy="median")
complaint_imputer = SimpleImputer(strategy="median")

user_scaler = StandardScaler()
complaint_scaler = StandardScaler()

X_user_search = user_imputer.fit_transform(
    search_frame[user_features]
)
X_complaint_search = complaint_imputer.fit_transform(
    search_frame[complaint_features]
)

X_user_search = user_scaler.fit_transform(
    X_user_search
).astype("float32")

X_complaint_search = complaint_scaler.fit_transform(
    X_complaint_search
).astype("float32")

# Give the user block and complaint block approximately equal total weight.
X_user_search /= np.sqrt(len(user_features))
X_complaint_search /= np.sqrt(len(complaint_features))

X_search = np.column_stack([
    X_user_search,
    X_complaint_search,
]).astype("float32")

del X_user_search, X_complaint_search
gc.collect()

print("Clustering matrix shape:", X_search.shape)

In [ ]:
def transform_cluster_frame(frame):
    """Apply fitted preprocessing and block weighting."""

    X_user = user_imputer.transform(
        frame[user_features]
    )
    X_complaint = complaint_imputer.transform(
        frame[complaint_features]
    )

    X_user = user_scaler.transform(
        X_user
    ).astype("float32")

    X_complaint = complaint_scaler.transform(
        X_complaint
    ).astype("float32")

    X_user /= np.sqrt(len(user_features))
    X_complaint /= np.sqrt(len(complaint_features))

    return np.column_stack([
        X_user,
        X_complaint,
    ]).astype("float32")

In [ ]:
metric_sample_size = min(
    METRIC_SAMPLE_SIZE,
    len(X_search),
)

metric_indices = rng.choice(
    len(X_search),
    size=metric_sample_size,
    replace=False,
)

X_metric = X_search[metric_indices]

diagnostic_rows = []

for k in K_CANDIDATES:
    seed_labels = []
    seed_metrics = []

    # Multiple seeds are used to evaluate stability.
    for seed in [42, 43, 44]:
        model = MiniBatchKMeans(
            n_clusters=k,
            random_state=seed,
            n_init=10,
            batch_size=4_096,
            max_iter=200,
            reassignment_ratio=0.01,
        )

        model.fit(X_search)
        labels = model.predict(X_metric)
        seed_labels.append(labels)

        silhouette = silhouette_score(
            X_metric,
            labels,
            sample_size=min(
                SILHOUETTE_SAMPLE_SIZE,
                len(X_metric),
            ),
            random_state=RANDOM_SEED,
        )

        ch_score = calinski_harabasz_score(
            X_metric,
            labels,
        )

        db_score = davies_bouldin_score(
            X_metric,
            labels,
        )

        cluster_shares = (
            pd.Series(labels)
            .value_counts(normalize=True)
        )

        seed_metrics.append({
            "inertia": model.inertia_,
            "silhouette": silhouette,
            "calinski_harabasz": ch_score,
            "davies_bouldin": db_score,
            "minimum_cluster_share": cluster_shares.min(),
        })

    pairwise_stability = [
        adjusted_rand_score(
            seed_labels[i],
            seed_labels[j],
        )
        for i, j in combinations(
            range(len(seed_labels)),
            2,
        )
    ]

    seed_metrics = pd.DataFrame(seed_metrics)

    diagnostic_rows.append({
        "k": k,
        "inertia": seed_metrics["inertia"].mean(),
        "silhouette": seed_metrics["silhouette"].mean(),
        "calinski_harabasz": (
            seed_metrics["calinski_harabasz"].mean()
        ),
        "davies_bouldin": (
            seed_metrics["davies_bouldin"].mean()
        ),
        "minimum_cluster_share": (
            seed_metrics["minimum_cluster_share"].mean()
        ),
        "stability_ari": np.mean(pairwise_stability),
    })

k_diagnostics = pd.DataFrame(diagnostic_rows)

display(
    k_diagnostics.style.format({
        "inertia": "{:,.0f}",
        "silhouette": "{:.4f}",
        "calinski_harabasz": "{:,.1f}",
        "davies_bouldin": "{:.4f}",
        "minimum_cluster_share": "{:.2%}",
        "stability_ari": "{:.4f}",
    })
)

In [ ]:
# Avoid solutions containing extremely small clusters.
valid_k = k_diagnostics[
    k_diagnostics["minimum_cluster_share"] >= 0.03
].copy()

if valid_k.empty:
    print(
        "Warning: every candidate K has a cluster below 3%. "
        "Using all K values for ranking."
    )
    valid_k = k_diagnostics.copy()

valid_k["rank_silhouette"] = (
    valid_k["silhouette"]
    .rank(ascending=False, method="min")
)

valid_k["rank_ch"] = (
    valid_k["calinski_harabasz"]
    .rank(ascending=False, method="min")
)

valid_k["rank_db"] = (
    valid_k["davies_bouldin"]
    .rank(ascending=True, method="min")
)

valid_k["rank_stability"] = (
    valid_k["stability_ari"]
    .rank(ascending=False, method="min")
)

valid_k["combined_rank"] = valid_k[
    [
        "rank_silhouette",
        "rank_ch",
        "rank_db",
        "rank_stability",
    ]
].sum(axis=1)

automatically_recommended_k = int(
    valid_k.sort_values(
        ["combined_rank", "k"]
    ).iloc[0]["k"]
)

chosen_k = (
    int(SELECTED_K)
    if SELECTED_K is not None
    else automatically_recommended_k
)

print("Automatically recommended K:", automatically_recommended_k)
print("K selected for final model:", chosen_k)

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(16, 9),
)

plot_specs = [
    ("inertia", "Inertia", False),
    ("silhouette", "Silhouette score", True),
    (
        "calinski_harabasz",
        "Calinski–Harabasz index",
        True,
    ),
    (
        "davies_bouldin",
        "Davies–Bouldin index",
        False,
    ),
    (
        "stability_ari",
        "Stability across seeds (ARI)",
        True,
    ),
    (
        "minimum_cluster_share",
        "Smallest cluster share",
        True,
    ),
]

for ax, (column, title, higher_is_better) in zip(
    axes.flat,
    plot_specs,
):
    sns.lineplot(
        data=k_diagnostics,
        x="k",
        y=column,
        marker="o",
        ax=ax,
    )

    ax.axvline(
        chosen_k,
        color="firebrick",
        linestyle="--",
        alpha=0.8,
        label=f"Selected K={chosen_k}",
    )

    ax.set_title(title)
    ax.set_xlabel("Number of clusters")

    if column == "minimum_cluster_share":
        ax.axhline(
            0.03,
            color="gray",
            linestyle=":",
            label="3% minimum",
        )

    ax.legend()

plt.suptitle(
    "K-means Cluster-Count Diagnostics",
    fontsize=16,
    y=1.02,
)

plt.tight_layout()
plt.show()

### full data

In [ ]:
persona_model = MiniBatchKMeans(
    n_clusters=chosen_k,
    random_state=RANDOM_SEED,
    n_init=10,
    batch_size=4_096,
    max_iter=200,
    reassignment_ratio=0.01,
)

# Incremental training over all eligible observations.
for epoch in range(FULL_TRAINING_EPOCHS):
    print(
        f"Full-data training epoch "
        f"{epoch + 1}/{FULL_TRAINING_EPOCHS}"
    )

    for start in range(
        0,
        len(eligible_positions),
        CHUNK_SIZE,
    ):
        batch_positions = eligible_positions[
            start:start + CHUNK_SIZE
        ]

        batch_frame = (
            df_segement.iloc[batch_positions]
            [cluster_features]
        )

        X_batch = transform_cluster_frame(
            batch_frame
        )

        persona_model.partial_fit(X_batch)

        del X_batch, batch_frame

    gc.collect()

In [ ]:
all_cluster_labels = np.full(
    len(df_segement),
    -1,
    dtype=np.int16,
)

for start in range(
    0,
    len(eligible_positions),
    CHUNK_SIZE,
):
    batch_positions = eligible_positions[
        start:start + CHUNK_SIZE
    ]

    batch_frame = (
        df_segement.iloc[batch_positions]
        [cluster_features]
    )

    X_batch = transform_cluster_frame(
        batch_frame
    )

    all_cluster_labels[batch_positions] = (
        persona_model.predict(X_batch)
    )

    del X_batch, batch_frame

cluster_series = pd.Series(
    all_cluster_labels,
    index=df_segement.index,
    dtype="Int16",
)

cluster_series = cluster_series.mask(
    cluster_series < 0
)

df_segement["persona_cluster"] = cluster_series

print(
    df_segement["persona_cluster"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
profile_df = df_segement.loc[
    df_segement["persona_cluster"].notna()
]

persona_profile = (
    profile_df
    .groupby(
        "persona_cluster",
        observed=True,
    )
    .agg(
        reviews=("recommendationid", "size"),
        median_hours_at_review=(
            "hours_at_review",
            "median",
        ),
        median_games_owned=(
            "num_games_owned",
            "median",
        ),
        median_author_reviews=(
            "author_num_reviews",
            "median",
        ),
        median_review_length=(
            "review_length_chars",
            "median",
        ),
        mean_factors_flagged=(
            "n_factors_flagged",
            "mean",
        ),
        d30_rate=(
            "still_playing_d30",
            "mean",
        ),
        steam_purchase_rate=(
            "steam_purchase",
            "mean",
        ),
        free_rate=(
            "received_for_free",
            "mean",
        ),
        early_access_rate=(
            "written_during_early_access",
            "mean",
        ),
    )
)

persona_profile["cluster_share"] = (
    persona_profile["reviews"]
    / persona_profile["reviews"].sum()
)

domain_columns = list(domain_groups)

domain_profile = (
    profile_df
    .groupby(
        "persona_cluster",
        observed=True,
    )[domain_columns]
    .mean()
)

persona_profile["dominant_complaint"] = (
    domain_profile.idxmax(axis=1)
    .str.replace(
        "domain_",
        "",
        regex=False,
    )
)

display(
    persona_profile[
        [
            "reviews",
            "cluster_share",
            "median_hours_at_review",
            "median_games_owned",
            "median_author_reviews",
            "median_review_length",
            "mean_factors_flagged",
            "dominant_complaint",
            "d30_rate",
            "steam_purchase_rate",
            "free_rate",
            "early_access_rate",
        ]
    ].style.format({
        "reviews": "{:,.0f}",
        "cluster_share": "{:.1%}",
        "median_hours_at_review": "{:,.1f}",
        "median_games_owned": "{:,.0f}",
        "median_author_reviews": "{:,.0f}",
        "median_review_length": "{:,.0f}",
        "mean_factors_flagged": "{:.2f}",
        "d30_rate": "{:.1%}",
        "steam_purchase_rate": "{:.1%}",
        "free_rate": "{:.1%}",
        "early_access_rate": "{:.1%}",
    })
)

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    domain_profile,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
)

plt.title(
    "Average Complaint-Domain Probability by Negative Reviewer Persona"
)
plt.xlabel("Complaint domain")
plt.ylabel("Persona cluster")
plt.tight_layout()
plt.show()

In [ ]:
weighted_centers = persona_model.cluster_centers_.copy()

n_user = len(user_features)

# Undo block weighting so centers return to standardized units.
weighted_centers[:, :n_user] *= np.sqrt(
    len(user_features)
)

weighted_centers[:, n_user:] *= np.sqrt(
    len(complaint_features)
)

center_profile = pd.DataFrame(
    weighted_centers,
    columns=cluster_features,
)

center_profile.index.name = "persona_cluster"

plt.figure(figsize=(15, 6))

sns.heatmap(
    center_profile,
    center=0,
    cmap="vlag",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
)

plt.title(
    "Standardized Characteristics of Negative-Reviewer Personas"
)
plt.xlabel("Clustering feature")
plt.ylabel("Persona cluster")
plt.tight_layout()
plt.show()

In [ ]:
visual_sample_size = min(
    VISUALIZATION_SAMPLE_SIZE,
    len(eligible_positions),
)

visual_positions = rng.choice(
    eligible_positions,
    size=visual_sample_size,
    replace=False,
)

visual_frame = (
    df_segement.iloc[visual_positions]
    [cluster_features]
)

X_visual = transform_cluster_frame(
    visual_frame
)

visual_labels = persona_model.predict(
    X_visual
)

pca = PCA(
    n_components=2,
    random_state=RANDOM_SEED,
)

X_pca = pca.fit_transform(X_visual)

pca_plot = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "persona_cluster": visual_labels.astype(str),
})

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=pca_plot,
    x="PC1",
    y="PC2",
    hue="persona_cluster",
    palette="tab10",
    alpha=0.35,
    s=18,
    linewidth=0,
)

plt.title(
    "PCA View of Negative-Reviewer Personas"
)
plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)"
)
plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)"
)

plt.legend(
    title="Persona cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()
plt.show()

### visualize pca

In [ ]:
visual_sample_size = min(
    VISUALIZATION_SAMPLE_SIZE,
    len(eligible_positions),
)

visual_positions = rng.choice(
    eligible_positions,
    size=visual_sample_size,
    replace=False,
)

visual_frame = (
    df_segement.iloc[visual_positions]
    [cluster_features]
)

X_visual = transform_cluster_frame(
    visual_frame
)

visual_labels = persona_model.predict(
    X_visual
)

pca = PCA(
    n_components=2,
    random_state=RANDOM_SEED,
)

X_pca = pca.fit_transform(X_visual)

pca_plot = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "persona_cluster": visual_labels.astype(str),
})

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=pca_plot,
    x="PC1",
    y="PC2",
    hue="persona_cluster",
    palette="tab10",
    alpha=0.35,
    s=18,
    linewidth=0,
)

plt.title(
    "PCA View of Negative-Reviewer Personas"
)
plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)"
)
plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)"
)

plt.legend(
    title="Persona cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()
plt.show()

In [ ]:
genre_columns = [
    column
    for column in df_segement.columns
    if column.startswith("genre_")
]

genre_profile = (
    profile_df
    .groupby(
        "persona_cluster",
        observed=True,
    )[genre_columns]
    .mean()
)

top_genres_by_cluster = {}

for cluster_id, row in genre_profile.iterrows():
    top_genres_by_cluster[int(cluster_id)] = (
        row.nlargest(5)
        .rename(lambda x: x.replace("genre_", ""))
        .to_dict()
    )

top_genres_by_cluster

In [ ]:
language_profile = (
    profile_df
    .groupby(
        ["persona_cluster", "language"],
        observed=True,
    )
    .size()
    .rename("reviews")
    .reset_index()
)

language_profile["within_cluster_share"] = (
    language_profile["reviews"]
    / language_profile.groupby(
        "persona_cluster"
    )["reviews"].transform("sum")
)

top_languages = (
    language_profile
    .sort_values(
        ["persona_cluster", "within_cluster_share"],
        ascending=[True, False],
    )
    .groupby(
        "persona_cluster",
        observed=True,
    )
    .head(5)
)

display(top_languages)